In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Change to your project directory
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    # Install required packages
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Comprehensive Comparison: FedAvg vs Centralized vs Ensemble Clustering

This notebook compares three training approaches on rotation-based non-IID CIFAR-10 data:
1. **Centralized Baseline**: Train a single model on all data combined
2. **Federated Averaging (FedAvg)**: Standard federated learning with model averaging
3. **Ensemble with Clustering**: Cluster-based federated learning with specialized models per cluster

All experiments use the same random seeds and data distribution for fair comparison.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset, ConcatDataset
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import copy
import random
from collections import defaultdict
import time

from training.ensemble_fl import EnsembleFedAvg
from training.utils import get_model, train, evaluate, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Configuration

Set all random seeds and hyperparameters for reproducibility.

## Step 1: Create Rotation-Based Dataset

Create a custom dataset with rotations to simulate non-IID data distribution.

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """Custom dataset that applies rotation to CIFAR-10 images"""
    def __init__(self, cifar_dataset, rotation_angle, transform=None):
        self.cifar_dataset = cifar_dataset
        self.rotation_angle = rotation_angle
        self.transform = transform
        self.base_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        
    def __len__(self):
        return len(self.cifar_dataset)
    
    def __getitem__(self, idx):
        image, label = self.cifar_dataset[idx]
        
        # Apply rotation
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        else:
            image = self.base_transform(image)
        
        return image, label

# Load CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## Step 2: Distribute Data to Clients

Assign clients to rotation clusters and distribute data.

In [ ]:
# Define rotation angles
rotation_angles = [0, 90, 180, 270]

# Assign clients to rotation clusters evenly
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"Client distribution across rotations:")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

# Split dataset among clients
samples_per_client = len(train_dataset) // CONFIG['num_clients']
all_indices = list(range(len(train_dataset)))
random.shuffle(all_indices)

train_subsets = []
for client_idx in range(CONFIG['num_clients']):
    start_idx = client_idx * samples_per_client
    end_idx = start_idx + samples_per_client
    
    if client_idx == CONFIG['num_clients'] - 1:
        end_idx = len(train_dataset)
    
    client_indices = all_indices[start_idx:end_idx]
    rotation_angle = client_rotation_labels[client_idx]
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset, rotation_angle)
    client_subset = Subset(rotated_dataset, client_indices)
    train_subsets.append(client_subset)

print(f"\nCreated {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")

# Create test dataset (no rotation)
test_rotated = RotatedCIFAR10Dataset(test_dataset, 0)
test_subset = Subset(test_rotated, list(range(len(test_dataset))))
test_loader = DataLoader(test_subset, batch_size=CONFIG['batch_size'], shuffle=False)

## Step 3: Centralized Baseline

Train a single model on all client data combined (simulating centralized learning).

In [ ]:
print("=" * 60)
print("TRAINING CENTRALIZED BASELINE")
print("=" * 60)

# Reset seed for fair comparison
set_seed(CONFIG['seed'])

# Combine all client datasets
combined_dataset = ConcatDataset(train_subsets)
combined_loader = DataLoader(
    combined_dataset, 
    batch_size=CONFIG['batch_size'], 
    shuffle=True
)

print(f"Combined dataset size: {len(combined_dataset)}")

# Initialize model
centralized_model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=CONFIG['num_classes'],
    pretrained=CONFIG['pretrained']
).to(device)

# Training setup
optimizer_cent = optim.SGD(
    centralized_model.parameters(), 
    lr=CONFIG['lr'], 
    momentum=CONFIG['momentum']
)
criterion_cent = nn.CrossEntropyLoss()

# Track metrics
centralized_train_losses = []
centralized_test_losses = []
centralized_test_accs = []
centralized_time = 0

# Training loop
start_time = time.time()
for epoch in range(1, CONFIG['centralized_epochs'] + 1):
    # Train
    centralized_model.train()
    epoch_loss = 0.0
    num_batches = 0
    
    for inputs, labels in combined_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer_cent.zero_grad()
        outputs = centralized_model(inputs)
        loss = criterion_cent(outputs, labels)
        loss.backward()
        optimizer_cent.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    centralized_train_losses.append(avg_train_loss)
    
    # Evaluate
    test_loss, test_acc = evaluate(centralized_model, test_loader, criterion_cent, device)
    centralized_test_losses.append(test_loss)
    centralized_test_accs.append(test_acc)
    
    print(f"Epoch {epoch}/{CONFIG['centralized_epochs']} - "
          f"Train Loss: {avg_train_loss:.4f}, "
          f"Test Loss: {test_loss:.4f}, "
          f"Test Acc: {test_acc:.4f}")

centralized_time = time.time() - start_time

print(f"\nCentralized Training Complete!")
print(f"Final Test Accuracy: {centralized_test_accs[-1]:.4f}")
print(f"Training Time: {centralized_time:.2f}s")

## Step 4: Federated Averaging (FedAvg)

Implement standard FedAvg algorithm with client sampling.

In [ ]:
print("\n" + "=" * 60)
print("TRAINING FEDERATED AVERAGING (FedAvg)")
print("=" * 60)

# Reset seed for fair comparison
set_seed(CONFIG['seed'])

# Initialize global model
fedavg_global_model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=CONFIG['num_classes'],
    pretrained=CONFIG['pretrained']
).to(device)

criterion_fedavg = nn.CrossEntropyLoss()

# Track metrics
fedavg_test_losses = []
fedavg_test_accs = []
fedavg_time = 0

def fedavg_aggregate(models_weights, client_sizes):
    """Aggregate client models using weighted averaging"""
    total_size = sum(client_sizes)
    avg_weights = copy.deepcopy(models_weights[0])
    
    for key in avg_weights.keys():
        avg_weights[key] = torch.zeros_like(avg_weights[key], dtype=torch.float)
        for i in range(len(models_weights)):
            avg_weights[key] += models_weights[i][key] * client_sizes[i]
        avg_weights[key] = torch.div(avg_weights[key], total_size)
    
    return avg_weights

# FedAvg training loop
start_time = time.time()
for round_num in range(1, CONFIG['fl_rounds'] + 1):
    print(f"\n--- Round {round_num}/{CONFIG['fl_rounds']} ---")
    
    # Sample clients
    num_selected = max(int(CONFIG['client_fraction'] * CONFIG['num_clients']), 1)
    selected_clients = random.sample(range(CONFIG['num_clients']), num_selected)
    
    # Store client updates
    client_weights = []
    client_sizes = []
    
    # Train selected clients
    global_weights = fedavg_global_model.state_dict()
    
    for client_idx in selected_clients:
        # Create local model copy
        local_model = get_model(
            model_name=CONFIG['model_name'],
            num_classes=CONFIG['num_classes'],
            pretrained=CONFIG['pretrained']
        ).to(device)
        local_model.load_state_dict(global_weights)
        
        # Local training
        optimizer_local = optim.SGD(
            local_model.parameters(), 
            lr=CONFIG['lr'], 
            momentum=CONFIG['momentum']
        )
        train_loader = DataLoader(
            train_subsets[client_idx], 
            batch_size=CONFIG['batch_size'], 
            shuffle=True
        )
        
        train(local_model, train_loader, optimizer_local, criterion_fedavg, device, 
              epochs=CONFIG['local_epochs'])
        
        # Store update
        client_weights.append(copy.deepcopy(local_model.state_dict()))
        client_sizes.append(len(train_subsets[client_idx]))
    
    # Aggregate updates
    aggregated_weights = fedavg_aggregate(client_weights, client_sizes)
    fedavg_global_model.load_state_dict(aggregated_weights)
    
    # Evaluate global model
    test_loss, test_acc = evaluate(fedavg_global_model, test_loader, criterion_fedavg, device)
    fedavg_test_losses.append(test_loss)
    fedavg_test_accs.append(test_acc)
    
    print(f"Round {round_num} - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

fedavg_time = time.time() - start_time

print(f"\nFedAvg Training Complete!")
print(f"Final Test Accuracy: {fedavg_test_accs[-1]:.4f}")
print(f"Training Time: {fedavg_time:.2f}s")

## Step 5: Ensemble with Clustering

Use the EnsembleFedAvg class to perform clustering-based federated learning.

In [ ]:
print("\n" + "=" * 60)
print("TRAINING ENSEMBLE WITH CLUSTERING")
print("=" * 60)

# Reset seed for fair comparison
set_seed(CONFIG['seed'])

# Initialize EnsembleFedAvg
ensemble_fl = EnsembleFedAvg(
    train_subsets=train_subsets,
    test_set=test_subset,
    num_clients=CONFIG['num_clients'],
    device=device,
    model_name=CONFIG['model_name'],
    pretrained=CONFIG['pretrained'],
    num_clusters=CONFIG['num_clusters'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    seed=CONFIG['seed']
)

# Track metrics
ensemble_test_losses = []
ensemble_test_accs = []
ensemble_time = 0

start_time = time.time()

# Step 1: Warmup phase (collect gradients)
print("\n--- Warmup Phase ---")
ensemble_fl.run_warmup(use_fedavg=False, local_epochs=CONFIG['warmup_epochs'])

# Step 2: Perform clustering on gradients
print("\n--- Clustering Phase ---")
# Use the last epoch gradients (or average across epochs)
gradients = ensemble_fl.get_client_gradients(average_across_rounds=True)
gradient_matrix = np.array([gradients[i] for i in range(CONFIG['num_clients'])])

# K-Means clustering
kmeans = KMeans(n_clusters=CONFIG['num_clusters'], random_state=CONFIG['seed'], n_init=10)
cluster_assignments = kmeans.fit_predict(gradient_matrix)

# Assign clusters to ensemble_fl
ensemble_fl.client_clusters = {i: cluster_assignments[i] for i in range(CONFIG['num_clients'])}

print(f"Clustering complete!")
for cluster_id in range(CONFIG['num_clusters']):
    count = np.sum(cluster_assignments == cluster_id)
    print(f"  Cluster {cluster_id}: {count} clients")

# Silhouette score
silhouette = silhouette_score(gradient_matrix, cluster_assignments)
print(f"Silhouette Score: {silhouette:.4f}")

# Step 3: Initialize ensemble
print("\n--- Initializing Ensemble ---")
ensemble_fl._initialize_ensemble()
print("Ensemble initialized!")

# Step 4: Train ensemble
print("\n--- Training Ensemble ---")
for round_num in range(1, CONFIG['fl_rounds'] + 1):
    loss, acc = ensemble_fl.train_ensemble_round(
        round_num=round_num,
        fraction=CONFIG['client_fraction'],
        local_epochs=CONFIG['local_epochs']
    )
    ensemble_test_losses.append(loss)
    ensemble_test_accs.append(acc)

ensemble_time = time.time() - start_time

print(f"\nEnsemble Training Complete!")
print(f"Final Test Accuracy: {ensemble_test_accs[-1]:.4f}")
print(f"Total Training Time: {ensemble_time:.2f}s")

## Step 6: Compare Clustering with Ground Truth

Analyze how well the gradient-based clustering matches the rotation-based data distribution.

In [ ]:
# Map rotation angles to cluster IDs
rotation_to_id = {0: 0, 90: 1, 180: 2, 270: 3}
true_clusters = np.array([rotation_to_id[angle] for angle in client_rotation_labels])
predicted_clusters = cluster_assignments

# Create confusion matrix
confusion_matrix = np.zeros((CONFIG['num_clusters'], CONFIG['num_clusters']), dtype=int)
for true_label, pred_label in zip(true_clusters, predicted_clusters):
    confusion_matrix[true_label, pred_label] += 1

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Pred {i}' for i in range(CONFIG['num_clusters'])],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(CONFIG['num_clusters'])])
plt.title('Confusion Matrix: True Rotation Clusters vs Predicted Clusters')
plt.ylabel('True Rotation Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.show()

# Calculate cluster purity
print("\nCluster Purity Analysis:")
for pred_cluster in range(CONFIG['num_clusters']):
    mask = predicted_clusters == pred_cluster
    if np.sum(mask) > 0:
        true_labels_in_cluster = true_clusters[mask]
        unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
        dominant_label = unique[np.argmax(counts)]
        purity = np.max(counts) / np.sum(mask)
        print(f"  Predicted Cluster {pred_cluster}:")
        print(f"    - Dominant rotation: {rotation_angles[dominant_label]}°")
        print(f"    - Purity: {purity:.2%}")
        print(f"    - Distribution: {dict(zip([rotation_angles[u] for u in unique], counts))}")

## Step 7: Comparison Results

Visualize and compare the performance of all three approaches.

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Test Accuracy over Time
ax1 = axes[0]
# Centralized (x-axis is epochs)
cent_x = np.arange(1, len(centralized_test_accs) + 1)
ax1.plot(cent_x, centralized_test_accs, 'o-', label='Centralized', linewidth=2, markersize=6)

# FedAvg (x-axis is rounds)
fedavg_x = np.arange(1, len(fedavg_test_accs) + 1)
ax1.plot(fedavg_x, fedavg_test_accs, 's-', label='FedAvg', linewidth=2, markersize=6)

# Ensemble (x-axis is rounds)
ensemble_x = np.arange(1, len(ensemble_test_accs) + 1)
ax1.plot(ensemble_x, ensemble_test_accs, '^-', label='Ensemble (Clustering)', linewidth=2, markersize=6)

ax1.set_xlabel('Epochs/Rounds')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Test Accuracy Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: Test Loss over Time
ax2 = axes[1]
ax2.plot(cent_x, centralized_test_losses, 'o-', label='Centralized', linewidth=2, markersize=6)
ax2.plot(fedavg_x, fedavg_test_losses, 's-', label='FedAvg', linewidth=2, markersize=6)
ax2.plot(ensemble_x, ensemble_test_losses, '^-', label='Ensemble (Clustering)', linewidth=2, markersize=6)

ax2.set_xlabel('Epochs/Rounds')
ax2.set_ylabel('Test Loss')
ax2.set_title('Test Loss Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final comparison table
print("\n" + "=" * 70)
print("FINAL COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Method':<25} {'Final Accuracy':<20} {'Training Time (s)':<20}")
print("-" * 70)
print(f"{'Centralized':<25} {centralized_test_accs[-1]:<20.4f} {centralized_time:<20.2f}")
print(f"{'FedAvg':<25} {fedavg_test_accs[-1]:<20.4f} {fedavg_time:<20.2f}")
print(f"{'Ensemble (Clustering)':<25} {ensemble_test_accs[-1]:<20.4f} {ensemble_time:<20.2f}")
print("=" * 70)

## Step 8: Statistical Analysis

Perform detailed analysis of convergence speed and stability.

In [ ]:
# Convergence analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy difference from centralized baseline
ax1 = axes[0, 0]
baseline_acc = centralized_test_accs[-1]
fedavg_diff = [acc - baseline_acc for acc in fedavg_test_accs]
ensemble_diff = [acc - baseline_acc for acc in ensemble_test_accs]

ax1.plot(fedavg_x, fedavg_diff, 's-', label='FedAvg', linewidth=2)
ax1.plot(ensemble_x, ensemble_diff, '^-', label='Ensemble', linewidth=2)
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='Centralized Baseline')
ax1.set_xlabel('Rounds')
ax1.set_ylabel('Accuracy Difference from Centralized')
ax1.set_title('Accuracy Gap vs Centralized Baseline')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Learning curves (first 10 epochs/rounds)
ax2 = axes[0, 1]
n_plot = min(10, len(centralized_test_accs), len(fedavg_test_accs), len(ensemble_test_accs))
ax2.plot(range(1, n_plot + 1), centralized_test_accs[:n_plot], 'o-', label='Centralized', linewidth=2)
ax2.plot(range(1, n_plot + 1), fedavg_test_accs[:n_plot], 's-', label='FedAvg', linewidth=2)
ax2.plot(range(1, n_plot + 1), ensemble_test_accs[:n_plot], '^-', label='Ensemble', linewidth=2)
ax2.set_xlabel('Epochs/Rounds')
ax2.set_ylabel('Test Accuracy')
ax2.set_title(f'Early Training (First {n_plot} Epochs/Rounds)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Improvement per round
ax3 = axes[1, 0]
fedavg_improvements = [0] + [fedavg_test_accs[i] - fedavg_test_accs[i-1] 
                              for i in range(1, len(fedavg_test_accs))]
ensemble_improvements = [0] + [ensemble_test_accs[i] - ensemble_test_accs[i-1] 
                                for i in range(1, len(ensemble_test_accs))]

ax3.plot(fedavg_x, fedavg_improvements, 's-', label='FedAvg', linewidth=2, alpha=0.7)
ax3.plot(ensemble_x, ensemble_improvements, '^-', label='Ensemble', linewidth=2, alpha=0.7)
ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax3.set_xlabel('Rounds')
ax3.set_ylabel('Accuracy Improvement from Previous Round')
ax3.set_title('Per-Round Improvement')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Final accuracy bar chart
ax4 = axes[1, 1]
methods = ['Centralized', 'FedAvg', 'Ensemble']
final_accs = [centralized_test_accs[-1], fedavg_test_accs[-1], ensemble_test_accs[-1]]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax4.bar(methods, final_accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax4.set_ylabel('Test Accuracy')
ax4.set_title('Final Test Accuracy Comparison')
ax4.set_ylim([min(final_accs) - 0.05, 1.0])
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, acc in zip(bars, final_accs):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.4f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics
print("\nConvergence Statistics:")
print("-" * 50)
print(f"Centralized:")
print(f"  - Best Accuracy: {max(centralized_test_accs):.4f}")
print(f"  - Accuracy Std Dev: {np.std(centralized_test_accs):.4f}")

print(f"\nFedAvg:")
print(f"  - Best Accuracy: {max(fedavg_test_accs):.4f}")
print(f"  - Accuracy Std Dev: {np.std(fedavg_test_accs):.4f}")
print(f"  - Avg Improvement/Round: {np.mean([x for x in fedavg_improvements if x > 0]):.4f}")

print(f"\nEnsemble (Clustering):")
print(f"  - Best Accuracy: {max(ensemble_test_accs):.4f}")
print(f"  - Accuracy Std Dev: {np.std(ensemble_test_accs):.4f}")
print(f"  - Avg Improvement/Round: {np.mean([x for x in ensemble_improvements if x > 0]):.4f}")
print(f"  - Clustering Silhouette: {silhouette:.4f}")

## Step 9: Per-Cluster Performance Analysis

Evaluate the ensemble model's performance on each rotation cluster separately.

In [ ]:
# Create test loaders for each rotation cluster
rotation_test_loaders = {}
for angle in rotation_angles:
    rotated_test = RotatedCIFAR10Dataset(test_dataset, angle)
    test_subset_rot = Subset(rotated_test, list(range(len(test_dataset))))
    rotation_test_loaders[angle] = DataLoader(
        test_subset_rot, 
        batch_size=CONFIG['batch_size'], 
        shuffle=False
    )

# Evaluate all models on each rotation
criterion_eval = nn.CrossEntropyLoss()

rotation_results = {
    'Centralized': {},
    'FedAvg': {},
    'Ensemble': {}
}

print("\nEvaluating models on rotated test sets...")
print("=" * 60)

for angle in rotation_angles:
    print(f"\nRotation: {angle}°")
    
    # Centralized
    loss, acc = evaluate(centralized_model, rotation_test_loaders[angle], criterion_eval, device)
    rotation_results['Centralized'][angle] = acc
    print(f"  Centralized: {acc:.4f}")
    
    # FedAvg
    loss, acc = evaluate(fedavg_global_model, rotation_test_loaders[angle], criterion_eval, device)
    rotation_results['FedAvg'][angle] = acc
    print(f"  FedAvg:      {acc:.4f}")
    
    # Ensemble
    loss, acc = ensemble_fl.evaluate_ensemble()
    # Need to evaluate on specific rotation
    from training.ensemble_model import EnsembleModel
    full_ensemble = EnsembleModel(ensemble_fl.cluster_models, ensemble_fl.feature_dim, 10)
    full_ensemble.classifier = ensemble_fl.ensemble_classifier
    full_ensemble.to(device)
    loss, acc = evaluate(full_ensemble, rotation_test_loaders[angle], criterion_eval, device)
    rotation_results['Ensemble'][angle] = acc
    print(f"  Ensemble:    {acc:.4f}")

# Visualize per-rotation performance
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(rotation_angles))
width = 0.25

bars1 = ax.bar(x - width, [rotation_results['Centralized'][a] for a in rotation_angles], 
               width, label='Centralized', alpha=0.8)
bars2 = ax.bar(x, [rotation_results['FedAvg'][a] for a in rotation_angles], 
               width, label='FedAvg', alpha=0.8)
bars3 = ax.bar(x + width, [rotation_results['Ensemble'][a] for a in rotation_angles], 
               width, label='Ensemble', alpha=0.8)

ax.set_xlabel('Rotation Angle')
ax.set_ylabel('Test Accuracy')
ax.set_title('Performance on Different Data Rotations')
ax.set_xticks(x)
ax.set_xticklabels([f'{a}°' for a in rotation_angles])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Calculate variance in performance across rotations
print("\nPerformance Variance Across Rotations:")
print("-" * 50)
for method in ['Centralized', 'FedAvg', 'Ensemble']:
    accs = [rotation_results[method][a] for a in rotation_angles]
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    print(f"{method:15} - Mean: {mean_acc:.4f}, Std: {std_acc:.4f}")

## Summary and Conclusions

### Key Findings:

1. **Centralized Baseline**: Provides upper bound performance with full data access
2. **FedAvg**: Standard federated learning, handles non-IID data with model averaging
3. **Ensemble with Clustering**: Leverages data heterogeneity by creating specialized models

### Performance Metrics Compared:
- **Final Test Accuracy**: Overall model performance
- **Convergence Speed**: How quickly each method reaches good performance
- **Robustness to Rotations**: Performance variance across different rotations
- **Training Time**: Computational efficiency

### Expected Observations:
- Centralized should achieve highest accuracy (has all data)
- FedAvg may struggle with extreme non-IID data (rotation-based)
- Ensemble clustering should adapt better to heterogeneous data by specialization
- Trade-offs between communication cost, privacy, and accuracy

### Next Steps:
- Experiment with different levels of non-IID data
- Test with more/fewer clusters
- Analyze communication costs
- Explore other clustering methods (hierarchical, DBSCAN)